# Milestone 6 - wiring the graph

**Why this is last.** Everything before this was ordinary Python. This milestone
is the LangGraph part: nodes, edges, a conditional edge, a cycle, and the
checkpointer that makes pausing for a human possible at all.

You import the finished node functions from `drill.nodes` - the lesson here is
the wiring, not the node bodies.

Read these first: `src/drill/graph.py` *after* you have tried, and the LangGraph
docs on [persistence](https://langchain-ai.github.io/langgraph/concepts/persistence/)
and [human-in-the-loop](https://langchain-ai.github.io/langgraph/concepts/human_in_the_loop/).

**How to work through this notebook.** Run the cells top to bottom. Each
exercise has a cell for you to fill in, then a cell of tests. Run the tests
first and watch them fail - read *what* they check, because the tests are the
specification. Then write code until they go green.

After you edit an exercise cell, **re-run it** before re-running its tests - the
tests use whatever is currently defined in the notebook.

In [ ]:
import ipytest
import pytest

# clean="test_*": before each test cell, forget test functions from earlier cells
# - but not other names starting with "Test", like milestone 2's TestCase class.
ipytest.autoconfig(clean="test_*")

from pathlib import Path

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command

from drill import nodes
from drill.nodes import route_after_tests
from drill.state import DrillState

## `build_drill_graph`

The shape you are aiming for:

```
START -> select_problem -> await_submission -> run_tests
                                ^                  |
                                |                  +-- passed -----> review --+
                                |                  |                          |
                                |                  +-- no tries left-> explain+
                                |                  |                          |
                                +------- hint <----+ otherwise                |
                                                                              v
                                                                    record -> END
```

Steps:

1. `StateGraph(DrillState)`.
2. `add_node(name, fn)` for each of the seven nodes in `drill.nodes`
   (`nodes.select_problem_node`, `nodes.await_submission_node`, ...).
3. `add_edge(START, "select_problem")` and the other plain edges.
4. `add_conditional_edges("run_tests", route_after_tests, {...})` where the dict
   maps each string your router can return to a node name. Pass it explicitly
   rather than relying on the names matching - a typo then fails at compile time
   instead of mid-session.
5. The cycle: an edge from `"hint"` back to `"await_submission"`. This is what
   makes it a graph rather than a pipeline.
6. Compile with a checkpointer. `InMemorySaver` is fine. This is **not**
   optional: `interrupt()` has to write the half-finished state somewhere before
   handing control back, so without a checkpointer there is nothing to resume
   from.

Two things the tests check that are easy to get wrong:

- Both `"review"` and `"explain"` must lead to `"record"`, so a session is
  recorded whether or not you solved it.
- `"record"` must lead to `END`.

In [ ]:
def build_drill_graph(checkpointer=None):
    """Build and compile the tutor graph."""
    # TODO: implement.
    raise NotImplementedError("Milestone 6: implement build_drill_graph")

## Look at it

Once it builds, draw it. If the picture does not match the diagram above, the
tests will not pass either - and the picture is quicker to read.

In [ ]:
from IPython.display import Markdown

Markdown(f"```mermaid\n{build_drill_graph().get_graph().draw_mermaid()}\n```")

## Shape tests

In [ ]:
%%ipytest -qq

@pytest.fixture
def built():
    return build_drill_graph()


def edges(graph) -> set[tuple[str, str]]:
    return {(e.source, e.target) for e in graph.get_graph().edges}


def test_all_seven_nodes_exist(built):
    names = set(built.get_graph().nodes)
    assert {"select_problem", "await_submission", "run_tests",
            "hint", "review", "explain", "record"} <= names


def test_the_happy_path_is_connected(built):
    assert {("__start__", "select_problem"),
            ("select_problem", "await_submission"),
            ("await_submission", "run_tests")} <= edges(built)


def test_run_tests_branches_three_ways(built):
    assert {("run_tests", "review"), ("run_tests", "hint"),
            ("run_tests", "explain")} <= edges(built)


def test_hint_loops_back_for_another_attempt(built):
    """The cycle. Without it this is a pipeline that gives up after one try."""
    assert ("hint", "await_submission") in edges(built)


def test_both_endings_are_recorded(built):
    assert {("review", "record"), ("explain", "record")} <= edges(built)


def test_record_ends_the_graph(built):
    assert ("record", "__end__") in edges(built)

## Running it

The shape tests only look at the graph. These actually run it - with a fake chat
model, so no Azure credentials are needed and every hint is a canned reply. The
fake is injected through the config, which is why `drill` passes its
dependencies around instead of using module globals.

In [ ]:
from drill.config import DrillSettings, LLMSettings
from drill.deps import Deps
from drill.progress import ProgressStore


class FakeChatModel:
    """A chat model that never touches the network - it always gives the same reply."""

    def invoke(self, messages, *args, **kwargs):
        class _Response:
            content = "a canned tutor reply"

        return _Response()


def make_deps(tmp: Path) -> Deps:
    settings = DrillSettings(
        llm=LLMSettings(), max_attempts=3, sandbox_timeout=15,
        progress_path=tmp / "progress.json",
        mlflow_tracking_uri=None, mlflow_experiment="notebook",
    )
    return Deps(chat_model=FakeChatModel(), settings=settings,
                progress=ProgressStore(settings.progress_path))


CORRECT = (
    "def two_sum(nums, target):\n"
    "    seen = {}\n"
    "    for i, n in enumerate(nums):\n"
    "        if target - n in seen:\n"
    "            return [seen[target - n], i]\n"
    "        seen[n] = i\n"
    "    return []\n"
)
WRONG = "def two_sum(nums, target):\n    return [0, 0]\n"

In [ ]:
%%ipytest -qq

@pytest.fixture
def built():
    return build_drill_graph()


@pytest.fixture
def deps(tmp_path):
    return make_deps(tmp_path)


def test_the_graph_pauses_for_a_submission(built, deps):
    out = built.invoke({"problem_id": "two_sum"}, config=deps.as_config("m6-a"))
    assert "__interrupt__" in out, "no checkpointer, or no interrupt in await_submission"
    assert out["__interrupt__"][0].value["problem_id"] == "two_sum"


def test_a_correct_submission_finishes(built, deps):
    config = deps.as_config("m6-b")
    built.invoke({"problem_id": "two_sum"}, config=config)
    out = built.invoke(Command(resume={"submission": CORRECT}), config=config)
    assert "__interrupt__" not in out
    assert out["verdict"] == "solved"


def test_a_wrong_submission_hints_and_asks_again(built, deps):
    config = deps.as_config("m6-c")
    built.invoke({"problem_id": "two_sum"}, config=config)
    out = built.invoke(Command(resume={"submission": WRONG}), config=config)
    assert "__interrupt__" in out
    assert out["__interrupt__"][0].value["attempt"] == 2


def test_running_out_of_attempts_explains(built, deps):
    config = deps.as_config("m6-d")
    built.invoke({"problem_id": "two_sum"}, config=config)
    for _ in range(3):
        out = built.invoke(Command(resume={"submission": WRONG}), config=config)
    assert "__interrupt__" not in out
    assert out["verdict"] == "explain"

## Try it: step through a session by hand

This is exactly what `drill start` does, minus the editor. Run each cell in turn
and watch the graph pause and resume. The `thread_id` in the config is the key
the checkpointer stores the paused state under - use a new one to start over.

In [ ]:
import tempfile

graph = build_drill_graph()
config = make_deps(Path(tempfile.mkdtemp())).as_config("by-hand")

out = graph.invoke({"problem_id": "two_sum"}, config=config)
out["__interrupt__"][0].value

In [ ]:
# A wrong answer: the graph runs the tests, hints, and pauses again.
out = graph.invoke(Command(resume={"submission": WRONG}), config=config)
out["__interrupt__"][0].value

In [ ]:
# The paused state lives in the checkpointer, not in your variables.
snapshot = graph.get_state(config)
snapshot.next, snapshot.values["attempts"]

In [ ]:
# Now solve it.
out = graph.invoke(Command(resume={"submission": CORRECT}), config=config)
out["verdict"]